In [1]:
import numpy as np
import pandas as pd
import streamlit as st 

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, roc_curve, roc_auc_score, accuracy_score

In [2]:
df = pd.read_csv('emails.csv')
df.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5728 entries, 0 to 5727
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    5728 non-null   str  
 1   spam    5728 non-null   int64
dtypes: int64(1), str(1)
memory usage: 8.6 MB


In [4]:
df['spam'].value_counts()

spam
0    4360
1    1368
Name: count, dtype: int64

In [5]:
df['text'][0]

"Subject: naturally irresistible your corporate identity  lt is really hard to recollect a company : the  market is full of suqgestions and the information isoverwhelminq ; but a good  catchy logo , stylish statlonery and outstanding website  will make the task much easier .  we do not promise that havinq ordered a iogo your  company will automaticaily become a world ieader : it isguite ciear that  without good products , effective business organization and practicable aim it  will be hotat nowadays market ; but we do promise that your marketing efforts  will become much more effective . here is the list of clear  benefits : creativeness : hand - made , original logos , specially done  to reflect your distinctive company image . convenience : logo and stationery  are provided in all formats ; easy - to - use content management system letsyou  change your website content and even its structure . promptness : you  will see logo drafts within three business days . affordability : your  ma

import nltk

from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

stop_words

In [6]:
df['text'][0].split()

['Subject:',
 'naturally',
 'irresistible',
 'your',
 'corporate',
 'identity',
 'lt',
 'is',
 'really',
 'hard',
 'to',
 'recollect',
 'a',
 'company',
 ':',
 'the',
 'market',
 'is',
 'full',
 'of',
 'suqgestions',
 'and',
 'the',
 'information',
 'isoverwhelminq',
 ';',
 'but',
 'a',
 'good',
 'catchy',
 'logo',
 ',',
 'stylish',
 'statlonery',
 'and',
 'outstanding',
 'website',
 'will',
 'make',
 'the',
 'task',
 'much',
 'easier',
 '.',
 'we',
 'do',
 'not',
 'promise',
 'that',
 'havinq',
 'ordered',
 'a',
 'iogo',
 'your',
 'company',
 'will',
 'automaticaily',
 'become',
 'a',
 'world',
 'ieader',
 ':',
 'it',
 'isguite',
 'ciear',
 'that',
 'without',
 'good',
 'products',
 ',',
 'effective',
 'business',
 'organization',
 'and',
 'practicable',
 'aim',
 'it',
 'will',
 'be',
 'hotat',
 'nowadays',
 'market',
 ';',
 'but',
 'we',
 'do',
 'promise',
 'that',
 'your',
 'marketing',
 'efforts',
 'will',
 'become',
 'much',
 'more',
 'effective',
 '.',
 'here',
 'is',
 'the',
 'l

In [7]:
import nltk

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to C:\Users\Electronica
[nltk_data]     Care\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [8]:
from nltk.corpus import stopwords

# 1. عرف المتغير الأول وتحقق من اللغة (arabic أو english)
stop_words = stopwords.words('arabic')


# 2. بعدين عرف الدالة
def remove_stop_words(text):
  words = text.split()
  words = [word for word in words if word not in stop_words]
  return ' '.join(words)


# 3. طبقها على الداتا فريم
df['new_text'] = df['text'].apply(remove_stop_words)

In [9]:
spam_words = " ".join(df[df['spam'] == 1]['new_text']).lower().split()
ham_words = " ".join(df[df['spam'] == 0]['new_text']).lower().split()

In [10]:
from collections import Counter

spam_counter = Counter(spam_words)
ham_counter = Counter(ham_words)    
spam_counter.most_common(100)

[('.', 19839),
 ('_', 13556),
 (',', 11514),
 ('the', 8975),
 ('-', 8514),
 ('to', 8165),
 ('and', 6517),
 ('of', 5629),
 ('you', 4920),
 ('a', 4695),
 (':', 4691),
 ('in', 3879),
 ('your', 3730),
 ('!', 3310),
 ('for', 3186),
 ('is', 2977),
 ('this', 2822),
 ('/', 2578),
 ("'", 2326),
 ('$', 2242),
 ('that', 1896),
 ('i', 1772),
 ('with', 1734),
 ('we', 1724),
 ('be', 1675),
 ('"', 1655),
 (')', 1598),
 ('are', 1590),
 ('or', 1588),
 ('it', 1584),
 ('on', 1582),
 ('(', 1514),
 ('from', 1471),
 ('will', 1429),
 ('our', 1404),
 ('not', 1382),
 ('subject:', 1368),
 ('have', 1352),
 ('s', 1333),
 ('*', 1300),
 ('as', 1201),
 ('=', 1182),
 ('by', 1038),
 ('com', 998),
 ('at', 988),
 ('?', 981),
 ('all', 979),
 ('1', 952),
 ('if', 869),
 ('business', 844),
 ('can', 830),
 ('company', 805),
 ('email', 804),
 ('an', 790),
 ('|', 789),
 ('here', 770),
 ('do', 767),
 ('information', 740),
 ('more', 727),
 ('my', 699),
 ('e', 698),
 ('no', 688),
 ('5', 687),
 ('money', 662),
 ('%', 659),
 ('2', 

In [11]:
ham_counter.most_common(100)


[('-', 96708),
 ('.', 72655),
 (',', 49629),
 ('the', 41135),
 ('/', 37733),
 (':', 37679),
 ('to', 33571),
 ('>', 21997),
 ('and', 20961),
 ('of', 18128),
 ('i', 15649),
 ('a', 15250),
 ('you', 14234),
 ('in', 14096),
 ('@', 13976),
 ('for', 13510),
 ('enron', 13382),
 ('ect', 11417),
 ('on', 11137),
 ('is', 9618),
 (')', 8988),
 ('vince', 8531),
 ("'", 8483),
 ('?', 8446),
 ('(', 8264),
 ('be', 7587),
 ('that', 7395),
 ('"', 7275),
 ('this', 7209),
 ('with', 7119),
 ('we', 6964),
 ('will', 6698),
 ('*', 6490),
 ('have', 6204),
 ('at', 5894),
 ('hou', 5569),
 ('your', 5544),
 ('from', 5275),
 ('s', 5173),
 ('it', 5104),
 ('as', 4974),
 ('2000', 4935),
 ('are', 4931),
 ('am', 4826),
 ('kaminski', 4770),
 ('by', 4474),
 ('com', 4444),
 ('please', 4386),
 ('subject:', 4360),
 ('if', 4324),
 ('subject', 4245),
 ('would', 4151),
 ('me', 3996),
 ('cc', 3889),
 ('re', 3825),
 ('j', 3693),
 ('_', 3678),
 ('or', 3630),
 ('can', 3425),
 ('pm', 3262),
 ('not', 3193),
 ('my', 3061),
 ('2001', 306

Preprocessing


In [12]:
df.duplicated().sum()


np.int64(33)

In [13]:
df['text'].duplicated().sum()


np.int64(33)

In [14]:
df.drop_duplicates(inplace=True,ignore_index=True)


In [15]:
df.isna().sum().max()


np.int64(0)

In [16]:
X = df['text']
y = df['spam']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((4556,), (1139,), (4556,), (1139,))

In [17]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

vectorizer = TfidfVectorizer(lowercase=True, stop_words='english', analyzer='word',ngram_range=(1,2),min_df=5,max_df=0.8,sublinear_tf=True)
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

In [18]:
X_train_vectorized_df = pd.DataFrame(X_train_vectorized.toarray(), columns=vectorizer.get_feature_names_out())
X_train_vectorized_df.describe()

,00,00 00,00 01,00 02,00 03,00 04,00 05,00 0500,00 06,00 0600,...,zip code,zip postal,ziplip,ziplip com,zipped,zipter,zipter oct,zipter rudi,zone,zzzz
count,4556.000000,4556.000000,4556.000000,4556.000000,4556.000000,4556.000000,4556.000000,4556.000000,4556.000000,4556.000000,...,4556.000000,4556.000000,4556.000000,4556.000000,4556.000000,4556.000000,4556.000000,4556.000000,4556.000000,4556.000000
mean,0.009140,0.001060,0.000294,0.000204,0.000144,0.000145,0.000124,0.000286,0.000161,0.000228,...,0.000177,0.000126,0.000564,0.000564,0.000102,0.000404,0.000123,0.000091,0.000238,0.000204
std,0.025166,0.010331,0.004788,0.004264,0.003685,0.003824,0.003076,0.004973,0.004059,0.004121,...,0.004594,0.003394,0.008096,0.008096,0.003127,0.005596,0.002637,0.002517,0.006343,0.006841
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,0.223288,0.246090,0.104338,0.111737,0.144555,0.135687,0.101131,0.135956,0.138986,0.119115,...,0.135933,0.122948,0.177818,0.177818,0.117087,0.130096,0.062051,0.074497,0.211951,0.341464


Modeling

In [19]:
log_reg = LogisticRegression(C=1, max_iter=1000, random_state=42,class_weight='balanced')
log_reg.fit(X_train_vectorized, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`m

In [20]:
y_valid_pred = cross_val_predict(log_reg, X_train_vectorized, y_train, cv=3, method='predict', n_jobs=-1)


print("Train Accuracy:", log_reg.score(X_train_vectorized, y_train))
print("Validation Accuracy:", accuracy_score(y_train, y_valid_pred))    
print('Precision:', precision_score(y_train, y_valid_pred))
print('Recall:', recall_score(y_train, y_valid_pred))
print('F1 Score:', f1_score(y_train, y_valid_pred))
confusion_matrix(y_train, y_valid_pred)

Train Accuracy: 0.9958296751536435
Validation Accuracy: 0.9883669885864794
Precision: 0.9610274579273693
Recall: 0.9917733089579525
F1 Score: 0.9761583445793972


array([[3418,   44],
       [   9, 1085]])

In [21]:
svm_model = LinearSVC(class_weight='balanced', random_state=42)

In [22]:
svm_model.fit(X_train_vectorized, y_train)


,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",'balanced'
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseud

In [23]:
y_valid_pred_svm = cross_val_predict(svm_model,X_train_vectorized, y_train, cv=3, method='predict', n_jobs=-1)

In [24]:
print("SVM Results")
print("Train Accuracy:", svm_model.score(X_train_vectorized, y_train))
print("Validation Accuracy:", accuracy_score(y_train, y_valid_pred_svm))
print("Precision:", precision_score(y_train, y_valid_pred_svm))
print("Recall:", recall_score(y_train, y_valid_pred_svm))
print("F1 Score:", f1_score(y_train, y_valid_pred_svm))
print("Confusion Matrix:")
print(confusion_matrix(y_train, y_valid_pred_svm))

SVM Results
Train Accuracy: 0.9995610184372257
Validation Accuracy: 0.992756804214223
Precision: 0.9880404783808647
Recall: 0.9817184643510055
F1 Score: 0.984869325997249
Confusion Matrix:
[[3449   13]
 [  20 1074]]


In [25]:
from sklearn.naive_bayes import MultinomialNB
nb_model = MultinomialNB()
nb_model.fit(X_train_vectorized, y_train)


,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [26]:
y_valid_pred_nb = cross_val_predict(nb_model,X_train_vectorized,y_train,cv=3,method='predict',n_jobs=-1)

In [27]:
print("Naive Bayes Results")
print("Train Accuracy:", nb_model.score(X_train_vectorized, y_train))
print("Validation Accuracy:", accuracy_score(y_train, y_valid_pred_nb))
print("Precision:", precision_score(y_train, y_valid_pred_nb))
print("Recall:", recall_score(y_train, y_valid_pred_nb))
print("F1 Score:", f1_score(y_train, y_valid_pred_nb))
print("Confusion Matrix:")
print(confusion_matrix(y_train, y_valid_pred_nb))

Naive Bayes Results
Train Accuracy: 0.9866110623353819
Validation Accuracy: 0.9620280948200176
Precision: 0.9989165763813651
Recall: 0.8427787934186471
F1 Score: 0.9142290530490828
Confusion Matrix:
[[3461    1]
 [ 172  922]]


In [28]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'SVM', 'Naive Bayes'],
    'Accuracy': [
        accuracy_score(y_train, y_valid_pred),
        accuracy_score(y_train, y_valid_pred_svm),
        accuracy_score(y_train, y_valid_pred_nb)
    ],
    'Precision': [
        precision_score(y_train, y_valid_pred),
        precision_score(y_train, y_valid_pred_svm),
        precision_score(y_train, y_valid_pred_nb)
    ],
    'Recall': [
        recall_score(y_train, y_valid_pred),
        recall_score(y_train, y_valid_pred_svm),
        recall_score(y_train, y_valid_pred_nb)
    ],
    'F1 Score': [
        f1_score(y_train, y_valid_pred),
        f1_score(y_train, y_valid_pred_svm),
        f1_score(y_train, y_valid_pred_nb)
    ]
})

results

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.988367,0.961027,0.991773,0.976158
1,SVM,0.992757,0.988040,0.981718,0.984869
2,Naive Bayes,0.962028,0.998917,0.842779,0.914229


In [29]:
plt.figure(figsize=(8,5))
sns.barplot(data=results, x='Model', y='F1 Score')
plt.title('Model Comparison Based on F1 Score')
plt.ylim(0, 1)
plt.show()

C:\Users\Electronica Care\AppData\Local\Temp\ipykernel_17568\2768145143.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [30]:
from sklearn.metrics import classification_report

y_test_pred_svm = svm_model.predict(X_test_vectorized)

print("Final Test Results - SVM")
print("Test Accuracy:", accuracy_score(y_test, y_test_pred_svm))
print("Precision:", precision_score(y_test, y_test_pred_svm))
print("Recall:", recall_score(y_test, y_test_pred_svm))
print("F1 Score:", f1_score(y_test, y_test_pred_svm))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_svm))
print("Classification Report:")
print(classification_report(y_test, y_test_pred_svm))

Final Test Results - SVM
Test Accuracy: 0.9956101843722563
Precision: 0.9890909090909091
Recall: 0.9927007299270073
F1 Score: 0.9908925318761385
Confusion Matrix:
[[862   3]
 [  2 272]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       865
           1       0.99      0.99      0.99       274

    accuracy                           1.00      1139
   macro avg       0.99      0.99      0.99      1139
weighted avg       1.00      1.00      1.00      1139



In [31]:
message = ["Congratulations! You have won a free prize, click here now"]

message_vectorized = vectorizer.transform(message)

prediction = svm_model.predict(message_vectorized)

if prediction[0] == 1:
    print("Spam Email")
else:
    print("Not Spam Email")

Spam Email


In [32]:
message = ["Hi, I hope you are doing well. Please find the meeting schedule attached."]

message_vectorized = vectorizer.transform(message)

prediction = svm_model.predict(message_vectorized)

if prediction[0] == 1:
    print("Spam Email")
else:
    print("Not Spam Email")

Not Spam Email


In [33]:
import pickle

In [34]:
pickle.dump(svm_model, open('spam_model.pkl', 'wb'))

pickle.dump(vectorizer, open('vectorizer.pkl', 'wb'))